In [2]:
# --- 0. Verify GPU + environment (run FIRST) ---
import os, sys, subprocess

print('Python', sys.version.split()[0])
print('cwd', os.getcwd())

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(gpu.stdout.strip() if gpu.stdout else 'NO GPU DETECTED - enable GPU P100 in Settings')

r = subprocess.run('curl -sI https://github.com -o /dev/null -w "%{http_code}"',
                   shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
code = (r.stdout or '').strip()
print('GitHub reachable, HTTP', code if code else 'FAIL')

Python 3.12.13
cwd /kaggle/working
Tesla P100-PCIE-16GB, 16384 MiB
GitHub reachable, HTTP 200


In [3]:
# --- 1. Install ComfyUI + wrappers (idempotent, shallow, no prompt hang) ---
import os, subprocess, time

def run(cmd, retries=3, timeout=900):
    env = dict(os.environ)
    env['GIT_TERMINAL_PROMPT'] = '0'   # never hang on a credential prompt
    for attempt in range(1, retries + 1):
        print(f"> {cmd}  (attempt {attempt}/{retries})")
        r = subprocess.run(cmd, shell=True, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, timeout=timeout)
        if r.stdout:
            print(r.stdout[-2000:])
        if r.returncode == 0:
            return True
        print(f"  ! attempt {attempt} failed (rc={r.returncode}); retrying in 5s...")
        time.sleep(5)
    return False

if not os.path.exists('ComfyUI'):
    run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI')
else:
    print('ComfyUI already present, skipping clone')

if os.path.exists('ComfyUI'):
    os.chdir('ComfyUI')
else:
    raise SystemExit('ERROR: ComfyUI clone failed - check Internet setting / GitHub access')

nodes = [
    'ComfyUI-CogVideoXWrapper',
    'ComfyUI-HunyuanVideoWrapper',
    'ComfyUI-VideoHelperSuite',
]
for repo in nodes:
    d = os.path.join('custom_nodes', repo)
    if not os.path.exists(d):
        run(f'git clone --depth 1 https://github.com/kijai/{repo} {d}')
    else:
        print(f'{repo} already present, skipping')

run('pip install -q -r requirements.txt')
run('pip install -q -r custom_nodes/ComfyUI-CogVideoXWrapper/requirements.txt', retries=1)
run('pip install -q -r custom_nodes/ComfyUI-HunyuanVideoWrapper/requirements.txt', retries=1)
print('comfyui + wrappers installed')

ComfyUI already present, skipping clone
ComfyUI-CogVideoXWrapper already present, skipping
ComfyUI-HunyuanVideoWrapper already present, skipping
ComfyUI-VideoHelperSuite already present, skipping
> pip install -q -r requirements.txt  (attempt 1/3)
> pip install -q -r custom_nodes/ComfyUI-CogVideoXWrapper/requirements.txt  (attempt 1/1)
> pip install -q -r custom_nodes/ComfyUI-HunyuanVideoWrapper/requirements.txt  (attempt 1/1)
comfyui + wrappers installed


In [4]:
# --- 2. Download model weights (only CogVideoX fits the free 16GB P100) ---
import os
os.makedirs('models/CogVideoX', exist_ok=True)
os.makedirs('models/text_encoders', exist_ok=True)

!pip install -q -U huggingface_hub   # prevents the interactive "update now?" prompt

!huggingface-cli download THUDM/CogVideoX-5b-I2V --local-dir models/CogVideoX --local-dir-use-symlinks False
!huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp8_e4m3fn.safetensors --local-dir models/text_encoders --local-dir-use-symlinks False
print('weights ready')


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

weights ready


In [5]:
# --- 3. Launch ComfyUI in the background ---
import subprocess, time

log = open('comfy.log', 'w')
proc = subprocess.Popen(['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188', '--disable-auto-launch'],
                        stdout=log, stderr=subprocess.STDOUT)
print('ComfyUI launching (pid', proc.pid, ')...')
time.sleep(25)
print(open('comfy.log').read()[-1500:])

ComfyUI launching (pid 640 )...
comfyui-frontend-package version: 1.49.6
[INFO] comfyui-workflow-templates version: 0.11.46
[INFO] comfyui-embedded-docs version: 0.5.10
[INFO] comfy-kitchen version: 0.2.31
[INFO] comfy-aimdo version: 0.4.13
[INFO] [Prompt Server] web root: /usr/local/lib/python3.12/dist-packages/comfyui_frontend_package/static
[INFO] Asset seeder disabled
[INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'
[INFO] NumExpr defaulting to 4 threads.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[INFO] 
Import times for custom nodes:
[INFO]    0.0 seconds: /kaggle/working/ComfyUI/custom_nodes/websocket_image_save.py
[INFO]    0.1 seconds: /kaggle/working/ComfyUI/custom_nodes/ComfyUI-CogVideoXWrapper


In [9]:
# --- 4. ngrok tunnel (ephemeral, no reserved domain needed) ---
import os, subprocess, time, json, urllib.request

os.environ['NGROK_AUTHTOKEN'] = '3IHH1tlbuuOlzVmJw6edmrUaG4z_3beUrzAJgeT4rXWoLZEKp'   # your real token

!pkill -9 ngrok 2>/dev/null; true
!rm -f ngrok ngrok-stable-linux-amd64.zip ngrok3.tgz
!curl -L -s -o ngrok3.tgz https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar xzf ngrok3.tgz
!./ngrok --version
!./ngrok config add-authtoken {os.environ['NGROK_AUTHTOKEN']}

tun = subprocess.Popen(['./ngrok','http','8188'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(4)
try:
    with urllib.request.urlopen('http://localhost:4040/api/tunnels', timeout=5) as r:
        t = json.load(r)
        url = t['tunnels'][0]['public_url']
        print('=== TUNNEL URL ===')
        print(url)
        print('===================')
except Exception as e:
    print('tunnel url fetch failed:', e)

ngrok version 3.39.11
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
=== TUNNEL URL ===
https://dullness-composure-sanctity.ngrok-free.dev


In [7]:
# --- 5. Keep-alive: poll ComfyUI so the session stays busy ---
import urllib.request, time, json
while True:
    try:
        with urllib.request.urlopen('http://localhost:8188/system_stats', timeout=5) as r:
            s = json.load(r)
            print('ComfyUI alive. GPU:', s.get('devices', [{}])[0].get('name', '?'))
    except Exception as e:
        print('ComfyUI check:', e)
    time.sleep(60)

ComfyUI alive. GPU: cuda:0 Tesla P100-PCIE-16GB : cudaMallocAsync
ComfyUI alive. GPU: cuda:0 Tesla P100-PCIE-16GB : cudaMallocAsync
ComfyUI alive. GPU: cuda:0 Tesla P100-PCIE-16GB : cudaMallocAsync
ComfyUI alive. GPU: cuda:0 Tesla P100-PCIE-16GB : cudaMallocAsync


KeyboardInterrupt: 